# 🔌 API Exploration Lab
Day 2 – Session 2

In this notebook we will authenticate, send API requests, inspect responses, and implement simple robustness patterns.

## 1) Setup & Credentials

In [ ]:
import os, json
from typing import Dict, Any
import time, random

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    print('⚠️ Set OPENAI_API_KEY in your environment to run live calls.')
else:
    print('✅ OPENAI_API_KEY is set.')

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
if not GOOGLE_API_KEY:
    print('⚠️ Set GOOGLE_API_KEY in your environment to run live calls.')
else:
    print('✅ GOOGLE_API_KEY is set.')

## 2) Helper: Safe API Generate Function with Retries

In [ ]:
import abc

class GenerativeAIClient(abc.ABC):

  @abc.abstractmethod
  def generate(self, prompt: str, **kwargs) -> str:
      ...

In [ ]:
from openai import OpenAI

class OpenAIClient(GenerativeAIClient):

  def generate(self, prompt: str, *, model: str='gpt-4o-mini', temperature: float=0.7, max_tokens: int=200, retries: int=3, backoff: float=0.8) -> str:
      """Call the chat completion API with basic retries and timing.
      Returns the model's answer as plain text.
      """
      if not OPENAI_API_KEY:
          raise ValueError("OPENAI_API_KEY is not set in the environment")

      client = OpenAI(api_key=OPENAI_API_KEY)
      response = client.chat.completions.create(
          model=model,
          messages=[
              {"role": "user", "content": prompt}
          ],
          temperature=temperature,
          max_tokens=max_tokens
      )
      if response is None:
          raise ValueError("No response from the API")

      choices = response.choices
      if not choices or len(choices) == 0:
          raise ValueError("Failed to get a valid response from the API")

      first_choice = choices[0]
      message = first_choice.message
      if message.role != "assistant":
          raise ValueError("Invalid message format in the response")

      if not message.content:
          reason = message.refusal
          raise ValueError("No content in the assistant's message: " + str(reason))
      
      return message.content

    

In [ ]:
from google import genai
from google.genai.types import GenerateContentConfig

class GoogleGenAIClient(GenerativeAIClient):

  def generate(self, prompt: str, *, model: str='gemini-2.5-flash', temperature: float=0.7, max_tokens: int=200, retries: int=3, backoff: float=0.8) -> str:
              
      """Call the chat completion API with basic retries and timing.
      Returns the model's answer as plain text.
      """

      if not isinstance(prompt, str):
          raise ValueError("Prompt should be a string")

      client = genai.Client()
      config=GenerateContentConfig(temperature=temperature, max_output_tokens=max_tokens)

      response = client.models.generate_content(model=model, contents=prompt, config=config)
      if response is None:
          raise ValueError("No response from the API")
      
      if not response.text:
          if response.candidates:
              reason = response.candidates[0].finish_reason
              if reason == "MAX_TOKENS":
                  raise ValueError("Response was cut off due to max tokens limit.")
              raise ValueError(f"No content in the response: {reason}")
          raise ValueError("Failed to get a valid response from the API")
      
      return response.text



In [ ]:
client = OpenAIClient()
response = client.generate("Hello, how are you?")
print(response)

In [ ]:
client = GoogleGenAIClient()
response = client.generate("Hello, how are you?", max_tokens=500)
print(response)

In [ ]:
client = GenerativeAIClient()


In [ ]:
from enum import Enum

class Provider(Enum):
    OPENAI = 'openai'
    GOOGLE = 'google'

class GenerativeAIClientFactory:
    
    @staticmethod
    def get_client(provider: Provider) -> GenerativeAIClient:
        if provider == Provider.OPENAI:
            return OpenAIClient()
        elif provider == Provider.GOOGLE:
            return GoogleGenAIClient()
        else:
            raise ValueError(f"Unknown provider: {provider}")

## 3) Compare Parameters

In [ ]:
prompt = 'Write three product taglines for a note-taking app.'

try:
    out1 = generate(prompt, temperature=0.2)
    out2 = generate(prompt, temperature=0.9)
    print('— Low temperature (0.2):\n', out1)
    print('\n— High temperature (0.9):\n', out2)

except Exception as e:
    raise # one would take the opportunity to do something more meaningful...

## 4) Add usage and latency metadata

Modify the function above so that it returns usage and latency.

In [ ]:
def generate(prompt: str, *, model: str='gpt-4o-mini', temperature: float=0.7, max_tokens: int=200, retries: int=3, backoff: float=0.8) -> Dict[str, Any]:
    """Call the chat completion API with basic retries and timing.
    Returns dict with text, usage (if available), and latency_ms.
    """
    ...

out = generate(prompt)
print('Usage:', json.dumps(out['usage']))
print('Latency (ms):', out['latency_ms'])

## 5) More exercises

1. Modify `generate` to accept a list of messages (system, user) and a stop sequence.
2. Add structured logging (JSON lines).
3. Try two prompts and compare responses for tone and length.